In [1]:
%load_ext autoreload
%autoreload 2

In [4]:
import numpy as np
from astrobridge.paper_pairing.core import CrossmatchBundle
from astrobridge.paper_pairing.filters import (
    MaxObjectsPerPaperFilter,
    MaxPapersPerObjectFilter,
    ObjectNameInTitleOrAbstractFilter,
    TitleKeywordFilter,
)
from astrobridge.paper_pairing.augmenters import PaperDownloadAugmenter
from astrobridge.assets import AssetManager

In [5]:
bundle = CrossmatchBundle.load("../data/paper_crossmatch_bundles/mmu_desi_edr_sv3")
print(bundle.summary())

CrossmatchBundle Summary
  hf_objects: 84555 rows, columns=['object_id', 'ra', 'dec']
  simbad_objects: 82600 rows, columns=['main_id', 'coo_bibcode']
  ads_papers: 10943 rows, columns=['bibcode', 'paper_title', 'abstract', 'doi', 'keyword', 'preprint_url']
  relationships: 358960 rows, columns=['hf_id', 'simbad_main_id', 'bibcode']


In [6]:
filter = ObjectNameInTitleOrAbstractFilter()
filter.apply(bundle, inplace=True)

In [7]:
asset_manager = AssetManager("../data/assets/papers")
augmenter = PaperDownloadAugmenter(asset_manager)
augmented_data = augmenter.augment(bundle, inplace=False)

Failed '1985A&A...150..302G': [error_permanent] HTTP 404
Failed '2016ATel.8647....1B': [error_permanent] HTTP 404
Failed '1997AJ....113.1483B': [error_permanent] HTTP 404
Failed '1985ApJ...290..496V': [error_permanent] HTTP 404
Failed '1981ApJ...247L...5M': [error_permanent] HTTP 404
Failed '1992ApJ...384..467V': [error_permanent] HTTP 404
Failed '1997A&A...327..550T': [error_permanent] HTTP 404
Failed '1987JApA....8..211K': [error_permanent] HTTP 404


In [8]:
print(augmented_data.summary())

CrossmatchBundle Summary
  hf_objects: 21 rows, columns=['object_id', 'ra', 'dec']
  simbad_objects: 16 rows, columns=['main_id', 'coo_bibcode']
  ads_papers: 31 rows, columns=['bibcode', 'paper_title', 'abstract', 'doi', 'keyword', 'preprint_url', 'download_status', 'download_message']
  relationships: 41 rows, columns=['hf_id', 'simbad_main_id', 'bibcode']


In [9]:
augmented_data.ads_papers['download_status'].value_counts()

download_status
success            23
error_permanent     8
Name: count, dtype: int64

In [10]:
mask = augmented_data.ads_papers['download_status'] == "error_permanent"
augmented_data.ads_papers['download_message'][mask]

3     HTTP 404
4     HTTP 404
11    HTTP 404
21    HTTP 404
23    HTTP 404
28    HTTP 404
29    HTTP 404
30    HTTP 404
Name: download_message, dtype: object